In [ ]:
!pip install datasets
!pip install faiss-cpu
!pip install transformers==4.46.3
!pip install keybert==0.8.5
!pip install langchain-community langchain-core langchain-huggingface


In [1]:
from datasets import load_dataset
dataset=load_dataset("CShorten/ML-ArXiv-Papers",split='train')
print(dataset)
dataset[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

In [2]:
import pandas as pd
df=pd.DataFrame(dataset)
df=df[['title','abstract']]
print(df)
print(df.shape)
df=df.head(15000)
print(df.shape)
print("null values:")
df.isnull().sum()

                                                    title  \
0                   Learning from compressed observations   
1       Sensor Networks with Random Links: Topology De...   
2       The on-line shortest path problem under partia...   
3         A neural network approach to ordinal regression   
4        Parametric Learning and Monte Carlo Optimization   
...                                                   ...   
117587  Detecting COVID-19 Conspiracy Theories with Tr...   
117588  Fair Feature Subset Selection using Multiobjec...   
117589  A Simple Duality Proof for Wasserstein Distrib...   
117590  Combined Learning of Neural Network Weights fo...   
117591  Adapting and Evaluating Influence-Estimation M...   

                                                 abstract  
0         The problem of statistical learning is to co...  
1         In a sensor network, in practice, the commun...  
2         The on-line shortest path problem is conside...  
3         Ordinal regressio

,0
title,0
abstract,0


In [3]:
df["paper_text"]=df["title"]+" "+df["abstract"]
print(df[["paper_text"]].head())


                                          paper_text
0  Learning from compressed observations   The pr...
1  Sensor Networks with Random Links: Topology De...
2  The on-line shortest path problem under partia...
3  A neural network approach to ordinal regressio...
4  Parametric Learning and Monte Carlo Optimizati...


In [4]:
print(type(df[["paper_text"]]))

<class 'pandas.core.frame.DataFrame'>


In [5]:
print(df['paper_text'].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

In [6]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()
sample_text=df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

In [7]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [8]:
embedding=model.encode(sample_text)
print(embedding.shape)

(384,)


In [9]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())
print(sample_embedding.shape)

(5, 384)


In [30]:
from sklearn.metrics.pairwise import cosine_similarity
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))[0][0]
  print(f"Paper 0 vs Paper {i}")
  print(f"Title: {df.iloc[i]['title']}")
  print(f"Similarity: {sim:.3f}\n")

Paper 0 vs Paper 1
Title: Sensor Networks with Random Links: Topology Design for Distributed
  Consensus
Similarity: 0.366

Paper 0 vs Paper 2
Title: The on-line shortest path problem under partial monitoring
Similarity: 0.335

Paper 0 vs Paper 3
Title: A neural network approach to ordinal regression
Similarity: 0.155

Paper 0 vs Paper 4
Title: Parametric Learning and Monte Carlo Optimization
Similarity: 0.374



In [11]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
  print("Loading saved embeddings")
  embeddings=np.load("paper_embeddings.npy")
else:
  print("generating embeddings")
  embeddings=model.encode(df["paper_text"].tolist(),batch_size=32,show_progress_bar=True)
  np.save("paper_embeddings.npy",embeddings)
  print("Embeddings saved successfully!")

Loading saved embeddings


In [12]:
import faiss
if os.path.exists("paper_faiss.index"):
  print("loading existing FAISS index")
  index=faiss.read_index("paper_faiss.index")
else:
  print("creating new FAISS index")
  faiss.normalize_L2(embeddings)
  index=faiss.IndexFlatIP(384)
  index.add(embeddings)
  faiss.write_index(index,"paper_faiss.index")
  print("FAISS index created successfully!")

loading existing FAISS index


In [13]:
print(index.ntotal)

15000


In [14]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score:", score)
    print("Title:",df.iloc[idx]["title"])
    print("Abstract:",df.iloc[idx]["abstract"][:500])
    print()
  return D,I

In [15]:
D,I=search_paper("deep learning for medical image analysis")

Similarity Score: 0.6807244
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity Score: 0.67092204
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract:   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for insta

In [16]:
from transformers import pipeline
summarizer=pipeline("summarization",model="facebook/bart-large-cnn")

In [17]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary[0]["summary_text"])

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.


In [18]:
def search_and_summarize(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score:", score)
    print("Title:",df.iloc[idx]["title"])
    print("Abstract:",df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
    print(summary[0]["summary_text"])
    print()

In [19]:
search_and_summarize("Deep learning in medical imaging",k=5)

Similarity Score: 0.73549855
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity Score: 0.6881736
Title: Classification of MRI data using Deep Learnin

In [20]:
from keybert import KeyBERT
kw_model=KeyBERT(model)

In [21]:
text=df.iloc[10466]["abstract"]
keywords=kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3), stop_words="english")
print(keywords)
for k,s in keywords:
  print(k)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]
tomographic imaging deep
imaging deep learning
learning medical imaging
imaging deep
medical imaging


In [22]:
from langchain_core.tools import tool

@tool
def search_paper_tool(query:str)-> str:
  """search research paper from the faiss database and return the most relevant result"""
  D,I=search_paper(query,k=3)
  results=[]
  for score,idx in zip(D[0],I[0]):
    results.append(f"Title: {df.iloc[idx]['title']}\nAbstract: {df.iloc[idx]['abstract'][:300]}...")
  return "\n\n".join(results)

print(search_paper_tool.invoke("deep learning for medical image analysis"))

Similarity Score: 0.6807244
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity Score: 0.67092204
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract:   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for insta

In [23]:
from langchain_huggingface import HuggingFacePipeline
llm=HuggingFacePipeline(pipeline=summarizer)

In [27]:
response=llm.invoke("Although depressive symptoms (DS) and physical activity (PA) have each been associated with social frailty (SF), their combined associations remain unclear. This study examined the associations of DS and objectively measured PA with SF in community-dwelling older adults.")
print(type(response))
print(response)

Your max_length is set to 142, but your input_length is only 53. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=26)


<class 'str'>
 depressive symptoms (DS) and physical activity (PA) have each been associated with social frailty (SF) Their combined associations remain unclear. This study examined the associations of DS and objectively measured PA with SF in community-dwelling older adults. The results were published in the Journal of the American Geriatrics Society.


In [26]:
def compare_papers(idx1, idx2, top_k_keywords=8):
  paper1=df.iloc[idx1]
  paper2=df.iloc[idx2]

  emb1=model.encode([paper1["paper_text"]])
  emb2=model.encode([paper2["paper_text"]])
  faiss.normalize_L2(emb1)
  faiss.normalize_L2(emb2)
  similarity=float(cosine_similarity(emb1,emb2)[0][0])

  summary1=summarizer(paper1["abstract"],max_length=100,min_length=30,do_sample=False)[0]["summary_text"]
  summary2=summarizer(paper2["abstract"],max_length=100,min_length=30,do_sample=False)[0]["summary_text"]

  kw1=kw_model.extract_keywords(paper1["abstract"],keyphrase_ngram_range=(1,2),stop_words="english",top_n=top_k_keywords)
  kw2=kw_model.extract_keywords(paper2["abstract"],keyphrase_ngram_range=(1,2),stop_words="english",top_n=top_k_keywords)
  kw1_set={k for k,_ in kw1}
  kw2_set={k for k,_ in kw2}
  shared=kw1_set & kw2_set
  unique1=kw1_set - kw2_set
  unique2=kw2_set - kw1_set

  if similarity>=0.75:
    verdict="Highly related — likely overlapping topic/methodology"
  elif similarity>=0.45:
    verdict="Loosely related — some shared themes"
  else:
    verdict="Largely unrelated"

  print("Paper 1:",paper1["title"])
  print("Summary 1:",summary1)
  print("Paper 2:",paper2["title"])
  print("Summary 2:",summary2)
  print("Cosine Similarity:",similarity)
  print("Shared Keywords:",shared)
  print("Unique to Paper 1:",unique1)
  print("Unique to Paper 2:",unique2)
  print("Verdict:",verdict)

  return {"similarity":similarity,"summary_1":summary1,"summary_2":summary2,
          "shared_keywords":shared,"unique_keywords_1":unique1,
          "unique_keywords_2":unique2,"verdict":verdict}

compare_papers(int(I[0][0]), int(I[0][1]))

Paper 1: A Perspective on Deep Imaging
Summary 1: The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.
Paper 2: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Summary 2: Training a deep convolutional neural network from scratch is difficult because it requires a large amount of labeled training data. A promising alternative is to fine-tune a CNN that has been pre-trained using, for instance, a large set of labeled natural images.
Cosine Similarity: 0.6056568026542664
Shared Keywords: set()
Unique to Paper 1: {'deep learning', 'medical imaging', 'imaging', 'image reconstruction', 'imaging develop', 'tomographic', 'imaging deep', 'tomographic imaging'}
Unique to 

{'similarity': 0.6056568026542664,
 'summary_1': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.',
 'summary_2': 'Training a deep convolutional neural network from scratch is difficult because it requires a large amount of labeled training data. A promising alternative is to fine-tune a CNN that has been pre-trained using, for instance, a large set of labeled natural images.',
 'shared_keywords': set(),
 'unique_keywords_1': {'deep learning',
  'image reconstruction',
  'imaging',
  'imaging deep',
  'imaging develop',
  'medical imaging',
  'tomographic',
  'tomographic imaging'},
 'unique_keywords_2': {'cnn adequate',
  'cnn trained',
  'cnns fine',
  'cnns trained',
  'deep cnn',
  'deep cnns',